# subQ_preparing.ipynb

Original Bi-CoT research notebook, recovered from the author’s AI_Study archive. Outputs, execution state, and machine metadata were removed; API credentials now come from `OPENAI_API_KEY`, and local paths now use `../data/workspace` relative to this notebook directory. See `docs/WORKFLOW.md` before selecting cells. Cells include experimental alternatives and data writes; this is not a single Run All pipeline.


In [ ]:
import os
import json
from collections import defaultdict

base_dir = "../data/workspace/5000_hotpot_devide"

# dependency_code 집계
depcode_stats = defaultdict(int)   # {dep_code: count}
id_to_depcode = {}                 # {id: dep_code}

for folder_id in os.listdir(base_dir):
    id_dir = os.path.join(base_dir, folder_id)
    sub_dir_0 = os.path.join(id_dir, "0")
    pjson_path = os.path.join(sub_dir_0, "p.json")
    if os.path.isfile(pjson_path):
        try:
            with open(pjson_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                dep_code = data.get("dependency_code", None)
                if dep_code is not None:
                    depcode_stats[dep_code] += 1
                    id_to_depcode[folder_id] = dep_code
        except Exception as e:
            print(f"⚠️ {pjson_path} 파일 읽기 오류: {e}")

print("🔍 dependency_code별 분포 (0 폴더 기준):")
for dep_code, count in depcode_stats.items():
    print(f"  {dep_code}: {count}개")

# id_to_depcode를 활용해 원하는 아이디 리스트도 쉽게 만들 수 있음

In [ ]:
# 위 코드에서 id_to_depcode 변수를 이미 만든 경우에 바로 사용 가능

def find_ids_by_depcode(dep_code_query):
    matching_ids = [folder_id for folder_id, code in id_to_depcode.items() if code == dep_code_query]
    print(f"\n'{dep_code_query}'를 가진 아이디 목록 ({len(matching_ids)}개):")
    for folder_id in matching_ids:
        print(folder_id)

# 예시 사용
find_ids_by_depcode("1-1")

In [ ]:
import os
import json

base_dir = "../data/workspace/5000_hotpot_devide"

def print_pjson_by_id(folder_id):
    pjson_path = os.path.join(base_dir, folder_id, "0", "p.json")
    if os.path.isfile(pjson_path):
        with open(pjson_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        # 보기 좋게 pretty print
        import pprint
        pprint.pprint(data, sort_dicts=False, width=120)
    else:
        print(f"❌ {pjson_path} 파일이 존재하지 않습니다.")

# 예시 사용
search_id = "5a88f2cc55429938390d3ffb"   # 원하는 아이디로 변경
print_pjson_by_id(search_id)

In [ ]:
import os
import json

def list_and_update_dependency_code_all(base_dir, folder_id, new_dep_code=None):
    """
    (1) 아이디 폴더 바로 아래의 0/p.json 출력/변경
    (2) 아이디 폴더의 각 subQ폴더/내부폴더/p.json도 모두 출력/변경
    """
    id_dir = os.path.join(base_dir, folder_id)
    if not os.path.isdir(id_dir):
        print(f"❌ {id_dir} 폴더가 존재하지 않습니다.")
        return

    found_any = False

    # (1) 최상위 0/p.json 체크
    top_pjson_path = os.path.join(id_dir, "0", "p.json")
    if os.path.isfile(top_pjson_path):
        with open(top_pjson_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        dep_code = data.get("dependency_code", None)
        print(f"0/p.json: dependency_code = {dep_code}")
        found_any = True
        if new_dep_code is not None:
            data["dependency_code"] = new_dep_code
            with open(top_pjson_path, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
            print(f"🔄 0/p.json의 dependency_code를 '{new_dep_code}'로 변경하여 저장했습니다.")

    # (2) 각 subQ폴더/내부폴더/p.json 체크
    subq_folders = sorted(os.listdir(id_dir))
    for subq_folder in subq_folders:
        subq_folder_path = os.path.join(id_dir, subq_folder)
        if os.path.isdir(subq_folder_path):
            # 최상위 0/p.json은 이미 처리했으니, subq_folder=="0"이면서 내부폴더가 없는 경우는 생략
            inner_folders = sorted(os.listdir(subq_folder_path))
            for inner_folder in inner_folders:
                inner_folder_path = os.path.join(subq_folder_path, inner_folder)
                if os.path.isdir(inner_folder_path):
                    pjson_path = os.path.join(inner_folder_path, "p.json")
                    if os.path.isfile(pjson_path):
                        with open(pjson_path, "r", encoding="utf-8") as f:
                            data = json.load(f)
                        dep_code = data.get("dependency_code", None)
                        print(f"{subq_folder}/{inner_folder}/p.json: dependency_code = {dep_code}")
                        found_any = True
                        if new_dep_code is not None:
                            data["dependency_code"] = new_dep_code
                            with open(pjson_path, "w", encoding="utf-8") as f:
                                json.dump(data, f, ensure_ascii=False, indent=2)
                            print(f"🔄 {subq_folder}/{inner_folder}/p.json의 dependency_code를 '{new_dep_code}'로 변경하여 저장했습니다.")
    if not found_any:
        print("❌ p.json 파일을 가진 폴더가 없습니다.")

# 사용 예시
base_dir = "../data/workspace/5000_hotpot_devide"
folder_id = "5a88f2cc55429938390d3ffb"

list_and_update_dependency_code_all(base_dir, folder_id)
# 바꾸고 싶으면 아래처럼 사용
list_and_update_dependency_code_all(base_dir, folder_id, new_dep_code="0")

In [ ]:
import json
from collections import Counter, defaultdict

hotpot_path = "../data/workspace/hotpot_qas_val.json"
retrieval_path = "../data/workspace/q01_beam3_topk3.json"

# 1. hotpot 파일도 한 줄씩 읽기 (jsonl 방식)
id2sp = {}
with open(hotpot_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            entry = json.loads(line)
            _id = entry["_id"]
            sp = entry.get("sp", [])
            id2sp[_id] = sp
        except Exception as e:
            print(f"⚠️ JSONDecodeError: {e}\n{line[:100]}")

# 2. retrieval 파일 분석 및 매칭 카운트 집계
match_counter = Counter()
id2nmatch = defaultdict(list)   # 아이디별로 n_match 리스트 저장
id2unique_titles = defaultdict(set)  # 아이디별 unique title 모음

with open(retrieval_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            entry = json.loads(line)
            _id = entry["_id"]
            candidate_chains = entry.get("candidate_chains", [])
            titles = []
            for chain in candidate_chains:
                for doc in chain:
                    t = doc.get("title")
                    if t is not None:
                        titles.append(t)
            unique_titles = set(titles)
            id2unique_titles[_id].update(unique_titles)  # unique title set에 추가

            sp_list = id2sp.get(_id, [])
            sp_in_titles = [sp for sp in sp_list if sp in unique_titles]
            n_match = len(sp_in_titles)
            n_sp = len(sp_list)

            # 매칭 결과 종류별로 카운트 (ex: "0/2", "1/2" ...)
            match_counter[f"{n_match}/{n_sp}"] += 1
            id2nmatch[_id].append(n_match)

            print(f"\nID: {_id}")
            print(f"  총 title 개수(중복 포함): {len(titles)}")
            print(f"  title 종류(중복 없이): {unique_titles} ({len(unique_titles)}종류)")
            print(f"  sp: {sp_list}")
            print(f"  sp 중 title에 포함된 개수: {n_match}/{n_sp} ({sp_in_titles})")
        except Exception as e:
            print(f"⚠️ JSONDecodeError (retrieval): {e}\n{line[:100]}")

# 3. 모두 0인 아이디 개수 세기
all_zero_ids = [k for k, v in id2nmatch.items() if len(v) == 3 and all(x == 0 for x in v)]

print("\n========== [전체 매칭 통계] ==========")
for match_type, count in sorted(match_counter.items()):
    print(f"  {match_type} : {count}개")

print(f"\n★★★ n_match가 모두 0인 아이디 개수: {len(all_zero_ids)}개")

# 4. 아이디별 unique title 종류 개수 통계
id_title_type_count = {k: len(v) for k, v in id2unique_titles.items()}
type_counter = Counter(id_title_type_count.values())
total_ids = len(id_title_type_count)
avg_type_count = sum(id_title_type_count.values()) / total_ids if total_ids else 0

print("\n========== [ID별 title 종류 개수 통계] ==========")
for n_type, cnt in sorted(type_counter.items()):
    print(f"  {n_type}종류 title 가진 ID : {cnt}개")
print(f"ID별 평균 title 종류 개수: {avg_type_count:.2f}")

In [ ]:
import os
import json
from collections import defaultdict

# 경로 세팅
json_path = "../data/workspace/q01_beam2_topk2_(3).json"
BASE_PATH = "../data/workspace/5000_hotpot_devide"  # 원하는 베이스 폴더로 변경하세요

# 아이디별로 엔트리 모으기
id2entries = defaultdict(list)

with open(json_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        entry = json.loads(line)
        _id = entry["_id"]
        id2entries[_id].append(entry)

total_ids = len(id2entries)
print(f"총 {total_ids}개 아이디의 데이터를 처리합니다.")

for idx, (_id, entries) in enumerate(id2entries.items(), 1):  # enumerate 시작값을 1로
    if len(entries) != 3:
        print(f"⚠️ {_id}는 엔트리가 {len(entries)}개입니다. (3개 아님, 건너뜀)")
        continue

    save_paths = [
        os.path.join(BASE_PATH, _id, "0", "q.json"),
        os.path.join(BASE_PATH, _id, "1", "0", "q.json"),
        os.path.join(BASE_PATH, _id, "1", "1", "q.json"),
    ]
    for i, entry in enumerate(entries):
        save_path = save_paths[i]
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(entry, f, ensure_ascii=False, indent=2)
    print(f"[{idx}/{total_ids}] {_id}: 0/q.json, 1/0/q.json, 1/1/q.json 저장 완료")

print("✅ 모든 아이디 처리 및 저장 완료")